# Calibration Study: Plots

Plots from the aggregated tables in this folder (`per_image.csv`, `per_condition.csv`, `thresholds.csv`, `geometry.json`). See [README.md](README.md) for what the columns mean and how they were computed.

Task: 2AFC, *which half is the unmodified image?* Chance is 50 %, the threshold is the 75 % point of the fitted psychometric function. `cutoff_cpd` is the highest spatial frequency rendered by the input (40 cpd = full resolution, halved per level); everything above was synthesized by the method.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Rectangle, Circle, Wedge

In [ ]:
per_image = pd.read_csv('per_image.csv')
per_cond = pd.read_csv('per_condition.csv')
thr = pd.read_csv('thresholds.csv')
geo = json.load(open('geometry.json'))

METHODS = ['foveated_noCE', 'foveated_CE', 'intensity_adjusted', 'oriented', 'phase_aligned']
LABEL = {'foveated_noCE': 'Foveated noCE', 'foveated_CE': 'Foveated CE', 'intensity_adjusted': 'Intensity Adjusted',
         'oriented': 'Oriented', 'phase_aligned': 'Phase Aligned'}
# colours as in the paper; marker and line style as secondary encoding
COLOR = {'foveated_noCE': '#bf0c4c', 'foveated_CE': '#6e072c', 'intensity_adjusted': '#deaf6b',
         'oriented': '#de8400', 'phase_aligned': '#377e8e'}
MARKER = {'foveated_noCE': 'o', 'foveated_CE': 's', 'intensity_adjusted': '^', 'oriented': 'D', 'phase_aligned': 'v'}
LS = {'foveated_noCE': '-', 'foveated_CE': '--', 'intensity_adjusted': ':', 'oriented': '-.', 'phase_aligned': '-'}
ECCS = [0, 8, 16, 24]
SCENES = sorted(per_image.scene.unique())

plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.25, 'font.size': 10, 'legend.frameon': False})

def weibull(cpd, l, k):
    """Psychometric function used for the thresholds (see build_dataset.py)."""
    return 1 - 0.5 * np.exp(-(((np.log2(40) - np.log2(cpd)) / l) ** k))

def cpd_axis(ax):
    ax.set_xscale('log', base=2)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:g}'))
    ax.set_xlabel('cutoff of the input [cpd]')

def ecc_title(ecc):
    return 'free viewing (0°)' if ecc == 0 else f'foveal mask {ecc}°'

print(f'{per_image.n_trials.sum()} trials, {len(per_image)} images x eccentricities, '
      f'{per_image.n_observers.max()} observers')

## 1. Number of Observations

Trials per image (scene × level) for every method and mask, after removing practice trials and repeated presentations (no outliers were removed). With 11 observers, each judging every image once per eccentricity, a complete cell has **11**. Cells with fewer are outlined. Grey: no trials. Orange line: the 75 % threshold of this method and mask (section 3), converted to a level; shaded: its 95 % bootstrap interval. Cells far from the line hardly affect the fitted threshold.

Where cells are missing or short:
- **Levels left out by design**: coarse levels that are obviously visible at low eccentricity; the extra coarse levels were only shown at high eccentricity. Level 0 (full resolution) is not part of the dataset (see README).
- **Last image of a block (sequence generator off by one)**: the generator dropped the last stimulus of each block of the stimulus list. The base block lists the scenes in the order `tree`, `classroom`, `house`, `face`, `fish`, `flat`, so its last entry is a `flat` image (level 5 of the last method); the extra-level blocks end with `house`. These images were (almost) never shown.
- **9 to 10 instead of 11 at the extra levels at 16° and 24°**: the extra levels were added after the first two observers.
- **Single trials**: the `house` trials at the extra levels are the last image of a block (see above), which only one observer's sequence happened to contain.
- **10 instead of 11 elsewhere**: a judgement lost to the early software bug that showed an image twice instead of the next one.

In [ ]:
levels = sorted(per_image.level.unique())
cmap = plt.get_cmap('Blues').copy()
cmap.set_bad('#d9d9d9')
fig, axes = plt.subplots(len(METHODS), 4, figsize=(15, 13), sharex=True, sharey=True, layout='constrained')
for r, m in enumerate(METHODS):
    for c, ecc in enumerate(ECCS):
        ax = axes[r, c]
        d = per_image[(per_image.method == m) & (per_image.mask_ecc_deg == ecc)]
        grid = d.pivot(index='scene', columns='level', values='n_trials').reindex(index=SCENES, columns=levels).values.astype(float)
        ax.imshow(np.ma.masked_invalid(grid), cmap=cmap, vmin=0, vmax=14, aspect='auto')
        ax.grid(False)
        for (i, j), v in np.ndenumerate(grid):
            if np.isnan(v):
                continue
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=8, color='white' if v > 8 else 'black',
                    fontweight='normal' if v == 11 else 'bold')
            if v != 11:
                ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fc='none', ec='black', lw=1.5))
        t = thr[(thr.method == m) & (thr.mask_ecc_deg == ecc)].iloc[0]
        lv = lambda cpd: np.log2(40 / cpd) - levels[0]   # cpd -> level -> column
        ax.axvspan(lv(t.ci95_high_boot_cpd), lv(t.ci95_low_boot_cpd), color='#de8400', alpha=0.25, lw=0)
        ax.axvline(lv(t.threshold_cpd), color='#de8400', lw=2.5)
        ax.set_xticks(range(len(levels)), [f'L{l}' for l in levels], fontsize=8)
        ax.set_yticks(range(len(SCENES)), SCENES, fontsize=8)
        if r == 0:
            ax.set_title(ecc_title(ecc))
        if c == 0:
            ax.set_ylabel(LABEL[m], fontsize=10, fontweight='bold')
plt.show()

totals = per_image.pivot_table(index='scene', columns='method', values='n_trials', aggfunc='sum')[METHODS]
totals.loc['all scenes'] = totals.sum()
print('Trials per scene and method (all levels and eccentricities):')
print(totals.rename(columns=LABEL).to_string())

## 2. Psychometric Functions

Proportion of correct answers per condition (pooled over the six scenes) with 95 % observer-bootstrap intervals, the fitted psychometric function and its 75 % threshold (open circle).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4.3), sharey=True)
x = np.geomspace(0.25, 40, 300)
for ax, ecc in zip(axes, ECCS):
    ax.axhline(0.5, color='0.4', lw=0.8)
    ax.axhline(0.75, color='0.4', lw=0.8, ls=':')
    for m in METHODS:
        d = per_cond[(per_cond.method == m) & (per_cond.mask_ecc_deg == ecc) & (per_cond.n_trials >= 20)]
        t = thr[(thr.method == m) & (thr.mask_ecc_deg == ecc)].iloc[0]
        ax.plot(x, weibull(x, t.weibull_l, t.weibull_k), color=COLOR[m], ls=LS[m], lw=1.6)
        ax.errorbar(d.cutoff_cpd, d.p_correct, yerr=[d.p_correct - d.ci95_low_boot, d.ci95_high_boot - d.p_correct],
                    fmt=MARKER[m], ms=5, color=COLOR[m], lw=1, capsize=2, alpha=0.85, label=LABEL[m])
        ax.plot(t.threshold_cpd, 0.75, 'o', ms=8, mfc='white', mec=COLOR[m], mew=2, zorder=5)
    cpd_axis(ax)
    ax.set_xlim(0.25, 45)
    ax.set_title(ecc_title(ecc))
axes[0].set_ylabel('proportion correct')
axes[0].set_ylim(0.3, 1.02)
axes[-1].legend(loc='lower left', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Thresholds Across Eccentricity

75 % thresholds with 95 % observer-bootstrap intervals. A lower threshold means a lower-resolution input suffices, i.e. the method hides the missing detail better. Points are slightly offset horizontally to avoid overlap.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))
for i, m in enumerate(METHODS):
    d = thr[thr.method == m].sort_values('mask_ecc_deg')
    xs = d.mask_ecc_deg + (i - 2) * 0.55
    ax.errorbar(xs, d.threshold_cpd, yerr=[d.threshold_cpd - d.ci95_low_boot_cpd, d.ci95_high_boot_cpd - d.threshold_cpd],
                fmt=MARKER[m], ls=LS[m], ms=7, lw=1.4, capsize=3, color=COLOR[m], label=LABEL[m])
ax.set_yscale('log', base=2)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: f'{v:g}'))
ax.set_xticks(ECCS, ['0 (free)', '8', '16', '24'])
ax.set_xlabel('foveal mask radius [deg]')
ax.set_ylabel('75 % threshold [cpd]')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# same thresholds as savings: fraction of pixels that must be rendered, relative to Foveated noCE
# (point estimates only; a ratio CI would need the joint bootstrap samples)
t = thr.pivot(index='method', columns='mask_ecc_deg', values='threshold_cpd').loc[METHODS]
pixels_vs_foveated = (t / t.loc['foveated_noCE']) ** 2
print('Rendered pixels needed relative to Foveated noCE (point estimates, (threshold / threshold_noCE)^2):')
print(pixels_vs_foveated.round(2).rename(index=LABEL).to_string())

## 4. Every Image

Proportion correct for every stimulus image (rows: scenes, columns: cutoff), per method and mask. Pink cells were not tested (or have fewer than 5 trials). Each cell is based on about 11 trials (one per observer), so single cells are noisy; look for consistent patterns across rows.

In [ ]:
levels = sorted(per_image.level.unique())
cmap = plt.get_cmap('Blues').copy()
cmap.set_bad('#f2a7c3')   # pink: not tested / fewer than 5 trials
fig, axes = plt.subplots(len(METHODS), 4, figsize=(15, 12.5), sharex=True, sharey=True, layout='constrained')
for r, m in enumerate(METHODS):
    for c, ecc in enumerate(ECCS):
        ax = axes[r, c]
        d = per_image[(per_image.method == m) & (per_image.mask_ecc_deg == ecc) & (per_image.n_trials >= 5)]
        grid = d.pivot(index='scene', columns='level', values='p_correct').reindex(index=SCENES, columns=levels)
        im = ax.imshow(np.ma.masked_invalid(grid.values.astype(float)), cmap=cmap, vmin=0.5, vmax=1.0, aspect='auto')
        ax.grid(False)
        ax.set_xticks(range(len(levels)), [f'{40 / 2**l:g}' for l in levels], fontsize=7, rotation=90)
        ax.set_yticks(range(len(SCENES)), SCENES, fontsize=8)
        if r == 0:
            ax.set_title(ecc_title(ecc))
        if c == 0:
            ax.set_ylabel(LABEL[m], fontsize=10, fontweight='bold')
        if r == len(METHODS) - 1:
            ax.set_xlabel('cutoff [cpd]')
fig.colorbar(im, ax=axes, shrink=0.4, label='proportion correct (0.5 = chance; pink = not tested)')
plt.show()

## 5. Which Scenes Are Harder?

Difference between each image's proportion correct and its condition average (same method, level and mask), per scene. Only conditions in the informative range (condition average between 0.6 and 0.95) are used; outside it every image is near chance or near perfect. Positive values: differences are easier to spot in this scene.

In [ ]:
d = per_image.merge(per_cond[['method', 'level', 'mask_ecc_deg', 'p_correct']],
                    on=['method', 'level', 'mask_ecc_deg'], suffixes=('', '_cond'))
d = d[d.method.isin(METHODS) & d.p_correct_cond.between(0.6, 0.95) & (d.n_trials >= 8)]
d['diff'] = d.p_correct - d.p_correct_cond
order = d.groupby('scene')['diff'].mean().sort_values().index

fig, ax = plt.subplots(figsize=(8, 4))
ax.axhline(0, color='0.4', lw=0.8)
ax.boxplot([d[d.scene == s]['diff'] for s in order], positions=range(len(order)), widths=0.5,
           showfliers=False, medianprops={'color': 'black'})
rng = np.random.default_rng(0)
for i, s in enumerate(order):
    v = d[d.scene == s]['diff']
    ax.plot(i + rng.uniform(-0.15, 0.15, len(v)), v, 'o', ms=3, color='#377e8e', alpha=0.4)
    ax.plot(i, v.mean(), 'D', ms=7, mfc='white', mec='black', zorder=5)
ax.set_xticks(range(len(order)), order)
ax.set_ylabel('proportion correct of the image\nminus the average over all six scenes')
ax.set_title(f'{len(d)} images from conditions with 0.6 to 0.95 correct (diamond: mean)', fontsize=10)
plt.tight_layout()
plt.show()

## 6. Viewing Geometry

The main display in its own plane, to scale in cm, seen from the observer at the left side (right-side sessions are mirrored). The observer's eye is 143 cm in front of the rim of the display (black square). For 8° the observer looks straight at the rim; for 16° and 24° the observer turned towards a fixation point on a helper display further to the side (red cross: direction of that point in the plane of the display), so the main display is seen at a slant.

- **Blue**: the image band (both halves), the only part of the screen with stimulus; light grey: screen margins without image.
- **Grey disc and ring**: the foveal mask (exact position and size from the Unity scene). Dark grey: fully opaque (inner circle); light grey ring: the soft edge, fading to fully transparent at the outer circle (2–5° wide, see the panel titles). Dotted: the mask beyond the screen. Although the display is slanted, both circles lie at an almost constant eccentricity (within 0.2°).
- **Dashed contours**: eccentricity in degrees for this eye position and gaze direction. Because the display is slanted, they are not circles around the fixation point.
- **Black dots**: the near corners of the display and their eccentricity.

In [ ]:
D = geo['observer']['viewing_distance_cm']
pitch = geo['display']['pixel_pitch_mm'] / 10
W_px, H_px = geo['display']['resolution_px']
eye_px = geo['observer']['eye_position_px'][0]

def ecc_deg(px, py, fix_px):
    """Eccentricity of screen points for the eye in front of the rim, looking towards fix_px (all in screen px)."""
    E = np.array([eye_px * pitch, H_px / 2 * pitch, -D])
    V = np.array([fix_px * pitch, H_px / 2 * pitch, 0.0])
    g = (V - E) / np.linalg.norm(V - E)
    P = np.stack([px * pitch, py * pitch, np.zeros_like(px)], -1) - E
    return np.degrees(np.arccos(np.clip(P @ g / np.linalg.norm(P, axis=-1), -1, 1)))

to_cm = lambda px, py: ((np.asarray(px) - eye_px) * pitch, (H_px / 2 - np.asarray(py)) * pitch)   # x from the rim, y up
gx, gy = np.meshgrid(np.linspace(-2400, W_px + 100, 700), np.linspace(-100, H_px + 100, 200))

fig, axes = plt.subplots(3, 1, figsize=(12, 12.5), sharex=True)
for ax, ecc in zip(axes, ['8', '16', '24']):
    cond, mask = geo['conditions'][ecc], geo['foveal_mask']['conditions'][ecc]
    fix = cond['fixation_px'][0]
    x0, y0 = to_cm(0, H_px)
    screen = Rectangle((x0, y0), W_px * pitch, H_px * pitch, fc='#eeeeee', ec='black', lw=1.2, zorder=0)
    ax.add_patch(screen)
    bx, by = to_cm(0, 2104)
    ax.add_patch(Rectangle((bx, by), W_px * pitch, 2048 * pitch, fc='#dbe9ec', ec='none', zorder=0))
    c = to_cm(*mask['centre_px'])
    r = {k: v * pitch for k, v in mask['radius_px'].items()}
    patches = [Wedge(c, r['fully_transparent_from'], 0, 360, width=r['fully_transparent_from'] - r['opaque_until'],
                     fc='0.55', alpha=0.3, ec='none'),
               Circle(c, r['opaque_until'], fc='0.55', alpha=0.6, ec='none'),
               Circle(c, r['opaque_until'], fc='none', ec='0.2', lw=0.9),
               Circle(c, r['fully_transparent_from'], fc='none', ec='0.2', lw=0.9)]
    for p in patches:
        ax.add_patch(p)
        p.set_clip_path(screen)
    for k in ('opaque_until', 'fully_transparent_from'):
        ax.add_patch(Circle(c, r[k], fc='none', ec='0.6', lw=0.8, ls=':'))
    cx, cy = to_cm(gx, gy)
    cs = ax.contour(cx, cy, ecc_deg(gx, gy, fix), levels=[10, 20, 30, 40, 50, 60], colors='0.3', linestyles='--', linewidths=0.7)
    ax.clabel(cs, fmt='%d°', fontsize=8)
    for py in (0, H_px):
        px_cm, py_cm = to_cm(0, py)
        ax.plot(px_cm, py_cm, 'o', color='black', ms=6, zorder=6)
    ax.annotate(f'corners {cond["tv_near_corners_ecc_deg"]:.1f}°', (x0 + 1.5, y0 + H_px * pitch - 4.5), fontsize=9)
    ax.plot(*to_cm(fix, H_px / 2), '+', color='#bf0c4c', ms=12, mew=2, zorder=6)
    ax.plot(0, 0, 's', color='black', ms=5, zorder=6)
    ax.annotate('eye 143 cm in front', (1, -4), fontsize=8)
    e_in = np.mean(mask['edge_eccentricity_deg']['opaque_until'])
    e_out = np.mean(mask['edge_eccentricity_deg']['fully_transparent_from'])
    ax.set_title(f'{ecc}° mask: gaze turned {cond["gaze_rotation_deg"]:.1f}°; mask fully opaque to {e_in:.1f}°, fading out until {e_out:.1f}°; '
                 f'far TV corners at {cond["tv_far_corners_ecc_deg"]:.1f}°', fontsize=10)
    ax.set_xlim(-60, 128); ax.set_ylim(-H_px / 2 * pitch - 4, H_px / 2 * pitch + 4)
    ax.set_aspect('equal'); ax.grid(False)
    ax.set_ylabel('cm (vertical)')
axes[-1].set_xlabel('cm from the rim of the display, in the plane of the display')
plt.tight_layout()
plt.show()